# Jimterpretability: Interactive Exploration

End-to-end demo using GPT-2 Small + `gpt2-small-res-jb` SAEs.

**Prerequisites:** `pip install -r requirements.txt` from the project root.

First run will download ~500MB of model weights.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
from jimterpretability.session import JimSession, SessionConfig
from jimterpretability.activations import extract_activations
from jimterpretability.features import aggregate_features
from jimterpretability.neuronpedia import NeuronpediaClient
from jimterpretability.interventions import steer, generate_with_interception, apply_rank1_weight_edit, restore_weights, compute_feature_direction

print('Imports OK')

## 1. Load Model + SAE

In [ ]:
config = SessionConfig.build(
    model_name='gpt2-small',
    layer=8,
    hook_type='resid_post',
    device='cpu',
    dtype='float32',
)
print(f'Loading {config.model_name} with SAE at {config.hook_point}...')
session = JimSession.create(config)
print(f'Session ready: {session.session_id}')

## 2. Concept Discovery — Staffordshire Terriers

In [ ]:
# In practice, use 50-400 prompts covering different angles of the concept
terrier_prompts = [
    'The Staffordshire Bull Terrier is a medium-sized breed.',
    'American Staffordshire Terriers are loyal and protective dogs.',
    'Staffies were originally bred for bull-baiting in England.',
    'The Staffordshire Terrier has a short, smooth coat.',
    'Pit bulls and Staffordshire Terriers share common ancestry.',
    'My Staffy loves playing fetch at the park.',
    'Staffordshire Bull Terriers are affectionate family pets.',
    'The American Staffordshire Terrier is recognized by the AKC.',
    'Staffordshires are known for their strength and determination.',
    'Training a Staffordshire Terrier requires consistent positive reinforcement.',
]

print(f'Processing {len(terrier_prompts)} prompts...')
feature_matrix = extract_activations(
    model=session.model,
    sae=session.sae,
    hook_point=session.config.hook_point,
    prompts=terrier_prompts,
    batch_size=4,
    aggregate_seq='mean',
)
print(f'Feature matrix: {feature_matrix.shape}')  # (n_prompts, n_features)

In [ ]:
stats = aggregate_features(feature_matrix, activation_threshold=0.0, top_k=20)

print('Top features by frequency:')
print(f'{"Rank":<5} {"Feature":<10} {"Frequency":<12} {"Mean Mag":<12} {"Max Mag":<10}')
print('-' * 55)
for i, s in enumerate(stats[:10]):
    print(f'{i+1:<5} {s.feature_idx:<10} {s.frequency:<12.3f} {s.mean_magnitude:<12.4f} {s.max_magnitude:<10.4f}')

## 3. Neuronpedia Labels

In [ ]:
import asyncio

top_idxs = [s.feature_idx for s in stats[:10]]

async def fetch_labels():
    async with NeuronpediaClient(
        model_id=session.neuronpedia_model_id,
        layer_id=session.neuronpedia_layer_id,
    ) as client:
        return await client.get_features_batch(top_idxs)

np_features = asyncio.run(fetch_labels())
label_map = {f.feature_idx: f.label for f in np_features}

print('Top features with Neuronpedia labels:')
for s in stats[:10]:
    label = label_map.get(s.feature_idx, 'unlabeled')
    print(f'  Feature {s.feature_idx:>6}  freq={s.frequency:.2f}  label={label}')

## 4. Activation Steering

In [ ]:
# Pick the top feature and steer toward it (amplify) or away from it (suppress)
TOP_FEATURE = stats[0].feature_idx
TEST_PROMPTS = ['The dog breed known as the Staffordshire']

print(f'Steering with feature {TOP_FEATURE}\n')

steered = steer(
    model=session.model,
    sae=session.sae,
    feature_idx=TOP_FEATURE,
    alpha=20.0,           # strong positive steering
    hook_point=session.config.hook_point,
    prompts=TEST_PROMPTS,
    max_new_tokens=40,
)
print('Steered output (alpha=+20):')
print(TEST_PROMPTS[0] + steered[0])

## 5. Output Interception

In [ ]:
output, was_intercepted = generate_with_interception(
    model=session.model,
    sae=session.sae,
    hook_point=session.config.hook_point,
    feature_idx=TOP_FEATURE,
    threshold=0.5,
    intercept_message='Sorry, I cannot talk about Staffordshire Terriers.',
    prompt='Tell me about Staffordshire Bull Terriers',
    max_new_tokens=60,
)

print(f'Was intercepted: {was_intercepted}')
print(f'Output: {output}')

## 6. Weight Editing (Permanent Suppression)

In [ ]:
# Project the feature direction out of MLP W_out at layer 8
feature_dir = compute_feature_direction(session.sae, TOP_FEATURE)
backup = apply_rank1_weight_edit(
    model=session.model,
    layer=session.config.layer,
    feature_direction=feature_dir,
    scale=1.0,
)
print(f'Weight edit applied (edit_id={backup["edit_id"]})')
print('Generate after weight edit:')

# After edit, the feature is harder to activate through that layer
tokens = session.model.to_tokens(['Tell me about Staffordshire Terriers'], prepend_bos=True)
with torch.no_grad():
    out_tokens = session.model.generate(tokens, max_new_tokens=40, verbose=False)
print(session.model.to_string(out_tokens[0, tokens.shape[1]:]))

# Restore original weights
restore_weights(session.model, session.config.layer, backup)
print('\nWeights restored.')

## 7. Feature Activation Distribution

In [ ]:
feature_activations = feature_matrix[:, TOP_FEATURE].numpy()

plt.figure(figsize=(8, 4))
plt.hist(feature_activations, bins=20, edgecolor='black')
plt.xlabel('Feature Activation Magnitude')
plt.ylabel('Number of Prompts')
plt.title(f'Feature {TOP_FEATURE} — Activation Distribution across {len(terrier_prompts)} Staffordshire Terrier Prompts')
plt.tight_layout()
plt.show()